# Tema 2 · Aprender a clasificar imágenes

Seis actividades sincronizadas con el PDF. Ejecuta en orden y guarda tu copia. Funciona con CPU y las bibliotecas del entorno del curso; no descarga datos ni modelos.

**Objetivos:** distinguir entrenamiento, validación y prueba; comparar una referencia con una red neuronal pequeña; interpretar una matriz de confusión y estudiar un cambio de condiciones. Las CNN y la transferencia se explican en la presentación: aquí ejecutamos regresión logística y una red densa (MLP), no una CNN.

Los datos son dibujos sintéticos de arandelas, placas y escuadras. No son fotografías ni acreditan rendimiento industrial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.dummy import DummyClassifier

NOMBRES = np.array(["Arandela", "Placa", "Escuadra"])
def dibujar(clase, radio, desplazamiento=(0, 0), luz=1.0, ruido=0.04, seed=0):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-16:16, -16:16]
    xx = xx - desplazamiento[0]
    yy = yy - desplazamiento[1]
    if clase == 0:
        distancia = np.sqrt(xx**2 + yy**2)
        mascara = (distancia < radio) & (distancia > radio * 0.5)
    elif clase == 1:
        mascara = (np.abs(xx) < radio) & (np.abs(yy) < radio * 0.7)
    else:
        mascara = ((np.abs(xx) < radio) & (np.abs(yy) < radio)
                   & ((xx < -radio * 0.25) | (yy > radio * 0.25)))
    imagen = np.where(mascara, 0.75, 0.15) * luz
    return np.clip(imagen + rng.normal(0, ruido, imagen.shape), 0, 1).astype(np.float32)

## Actividad 1 · Datos y etiquetas (10 min)

Generamos 60 piezas distintas por clase y tres capturas simuladas por pieza. Cada pieza tiene un identificador. Las tres capturas comparten geometría: no son tres piezas independientes. Predice qué variaciones deberían conservar su clase.

In [ ]:
rng = np.random.default_rng(12)
imagenes, etiquetas, grupos = [], [], []
for clase in range(3):
    for pieza in range(60):
        radio = rng.uniform(7, 12)
        grupo = clase * 60 + pieza
        for vista in range(3):
            imagenes.append(dibujar(clase, radio,
                desplazamiento=tuple(rng.integers(-3, 4, size=2)),
                luz=rng.uniform(0.8, 1.2), seed=grupo * 3 + vista))
            etiquetas.append(clase)
            grupos.append(grupo)
imagenes = np.stack(imagenes)
y = np.array(etiquetas)
grupos = np.array(grupos)
X = imagenes.reshape(len(imagenes), -1)
print("Imágenes:", imagenes.shape, "Entrada por imagen:", X.shape[1])
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for clase in range(3):
    ids = np.flatnonzero(y == clase)[::15][:4]
    for ax, idx in zip(axes[clase], ids):
        ax.imshow(imagenes[idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{NOMBRES[y[idx]]} · {grupos[idx]}")
        ax.axis("off")
plt.tight_layout()
plt.show()

**Explica:** ¿qué aprende el modelo: el nombre escrito, los píxeles o una regla que tú programas? ¿Una foto de otra cara de la misma pieza sería completamente independiente?

**Tu respuesta:** …

## Actividad 2 · Separar antes de aprender (10 min)

Reservamos aproximadamente 60 % de piezas para entrenamiento, 20 % para validación y 20 % para prueba. La separación es por pieza, no por captura. Los porcentajes por clase pueden variar: comprueba los recuentos.

In [ ]:
split1 = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=7)
train, resto = next(split1.split(X, y, groups=grupos))
split2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=8)
a, b = next(split2.split(X[resto], y[resto], groups=grupos[resto]))
val, test = resto[a], resto[b]
assert not (set(grupos[train]) & set(grupos[val]))
assert not (set(grupos[train]) & set(grupos[test]))
assert not (set(grupos[val]) & set(grupos[test]))
for nombre, idx in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    print(nombre, len(idx), "imágenes; por clase:", np.bincount(y[idx], minlength=3))
    assert len(np.unique(y[idx])) == 3

**Explica:** ¿por qué sería engañoso repartir las capturas de una misma pieza entre entrenamiento y prueba? Conserva esta separación para todas las comparaciones.

**Tu respuesta:** …

## Actividad 3 · Una referencia antes de la red (12 min)

El modelo más simple siempre predice la clase más frecuente del entrenamiento. La regresión logística aprende pesos de los píxeles para separar clases. El escalado se ajusta solo con entrenamiento mediante un pipeline.

In [ ]:
referencia = DummyClassifier(strategy="most_frequent")
referencia.fit(X[train], y[train])
lineal = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500))
lineal.fit(X[train], y[train])
for nombre, modelo in [("Mayoría", referencia), ("Lineal", lineal)]:
    print(nombre, "acierto validación:", round(accuracy_score(y[val], modelo.predict(X[val])), 3))

**Predice y modifica:** repite con `C=0.01` y `C=1`. Un C menor impone mayor regularización. Compara únicamente en validación y registra el cambio. No consultes prueba todavía.

**Tu respuesta:** …

## Actividad 4 · Una red neuronal pequeña (15 min)

La MLP conecta píxeles con 32 neuronas ocultas y tres salidas. Entrenamos durante 50 épocas; una época recorre todos los ejemplos de entrenamiento. Evaluamos cada época sin ajustar pesos con validación. El escalador se ajusta solo en entrenamiento.

Las curvas ayudan a observar tendencias; no todos los conjuntos muestran sobreajuste de forma clara.

In [ ]:
escalador = StandardScaler().fit(X[train])
Xt = escalador.transform(X[train])
Xv = escalador.transform(X[val])
red = MLPClassifier(hidden_layer_sizes=(32,), activation="relu",
                    solver="adam", alpha=0.01, batch_size=64, random_state=10)
historial = []
for epoca in range(50):
    red.partial_fit(Xt, y[train], classes=np.arange(3))
    historial.append([red.loss_, red.score(Xt, y[train]), red.score(Xv, y[val])])
historial = np.array(historial)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(np.arange(1, 51), historial[:, 0])
axes[0].set(xlabel="Época", ylabel="Pérdida de entrenamiento")
axes[1].plot(np.arange(1, 51), historial[:, 1], label="Entrenamiento")
axes[1].plot(np.arange(1, 51), historial[:, 2], label="Validación")
axes[1].set(xlabel="Época", ylabel="Proporción de aciertos", ylim=(0, 1.05))
axes[1].legend()
plt.tight_layout()
plt.show()
print("Acierto validación MLP:", round(red.score(Xv, y[val]), 3))

**Comprueba:** la pérdida y el acierto no son la misma magnitud. ¿Mejoran a la vez? Como ampliación, prueba 8 neuronas o menos datos de entrenamiento y compara en validación. Vuelve a ejecutar desde aquí para conservar coherencia.

**Tu respuesta:** …

## Actividad 5 · Cerrar decisiones y evaluar (12 min)

Elegimos entre el modelo lineal y la MLP usando solo validación. En empate, preferimos el lineal por sencillez. Después evaluamos una vez en prueba. No vuelvas a elegir parámetros por lo que veas en prueba: si lo haces, necesitarás un nuevo conjunto independiente.

In [ ]:
acierto_lineal = lineal.score(X[val], y[val])
acierto_red = red.score(Xv, y[val])
usar_red = acierto_red > acierto_lineal
nombre_elegido = "MLP" if usar_red else "Lineal"
def predecir(datos):
    return red.predict(escalador.transform(datos)) if usar_red else lineal.predict(datos)
pred = predecir(X[test])
print("Modelo elegido en validación:", nombre_elegido)
print(classification_report(y[test], pred, target_names=NOMBRES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y[test], pred, labels=np.arange(3),
                                       display_labels=NOMBRES, cmap="Blues")
plt.title("Prueba: filas reales, columnas predichas")
plt.show()
fallos = np.flatnonzero(pred != y[test])
if len(fallos):
    fig, axes = plt.subplots(1, min(4, len(fallos)), figsize=(10, 3), squeeze=False)
    for ax, posicion in zip(axes[0], fallos[:4]):
        idx = test[posicion]
        ax.imshow(imagenes[idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Real: {NOMBRES[y[idx]]}\nPred.: {NOMBRES[pred[posicion]]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Sin errores en esta prueba sintética; no implica éxito en fotografías reales.")

**Interpreta:** describe dos errores si existen. Si no hay errores, documenta dos limitaciones del conjunto y analiza los fallos de la actividad siguiente. En el informe, *precision* indica qué fracción de las predicciones de una clase era correcta; *recall*, qué fracción de sus ejemplos reales fue recuperada.

**Tu respuesta:** …

## Actividad 6 · Cambiar las condiciones (13 min)

Experimento diagnóstico posterior a la evaluación: desplazamos las imágenes de prueba 6 píxeles hacia la derecha y 4 hacia abajo. Rellenamos con el fondo; no hay retorno circular. Algunas piezas pueden quedar recortadas. Conservamos el modelo: no lo entrenamos de nuevo. Este experimento no es una nueva prueba independiente.

In [ ]:
desplazadas = np.full_like(imagenes[test], 0.15)
desplazadas[:, 4:, 6:] = imagenes[test, :-4, :-6]
pred_cambio = predecir(desplazadas.reshape(len(test), -1))
print("Acierto original:", round(accuracy_score(y[test], pred), 3))
print("Acierto tras desplazar:", round(accuracy_score(y[test], pred_cambio), 3))
fallos_cambio = np.flatnonzero(pred_cambio != y[test])
seleccion = fallos_cambio[:4] if len(fallos_cambio) else np.arange(4)
fig, axes = plt.subplots(2, len(seleccion), figsize=(10, 5), squeeze=False)
for col, pos in enumerate(seleccion):
    for fila, datos in enumerate([imagenes[test], desplazadas]):
        axes[fila, col].imshow(datos[pos], cmap="gray", vmin=0, vmax=1)
        axes[fila, col].axis("off")
    axes[0, col].set_title(f"Real: {NOMBRES[y[test[pos]]]}")
    axes[1, col].set_title(f"Pred.: {NOMBRES[pred_cambio[pos]]}")
plt.tight_layout()
plt.show()

**Discute:** ¿por qué este cambio puede perjudicar a un modelo que recibe píxeles aplanados? ¿Cómo organizarías una nueva captura y una evaluación independiente? Una CNN puede aprovechar estructura local, pero no garantiza invariancia frente a cualquier desplazamiento.

**Tu respuesta:** …

## Entrega y autoevaluación

Incluye: recuentos de las particiones, comparación en validación, curvas, matriz de confusión y dos errores o limitaciones comentadas. Explica la diferencia entre aprender pesos y elegir hiperparámetros. Propón cómo recopilarías fotografías reales para comprobar la utilidad industrial.

**Extensión conceptual:** en transferencia, un extractor preentrenado produce características y entrenamos una nueva cabeza. No hemos descargado ni ejecutado ese extractor en esta práctica. La elección de CNN, datos preentrenados y transformación de entrada exige una práctica específica posterior.